In [3]:
#"This file was used to apply a Bayesian model on the architecture of an LSTM and then compare the results."

# Imports
import math
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


df = pd.read_csv('GOOGL_historical_data.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'GOOGL_historical_data.csv'

In [ ]:
# -----------------------------
# 1) Prepare data
# -----------------------------
# Expected: df with columns 'Date' and 'Close'
assert 'Close' in df.columns, "DataFrame df must contain the column 'Close'."

close = df['Close'].values.astype(np.float32).reshape(-1, 1)

# Parameters
look_back = 10         # Window length
split_percent = 0.8    # 80/20 split
split_idx = int(len(close) * split_percent)

# Scaling: fit only on training set
scaler = MinMaxScaler()
close_train = scaler.fit_transform(close[:split_idx])
close_test  = scaler.transform(close[split_idx:])

def make_windows(arr, L):
    X, y = [], []
    for i in range(len(arr) - L):
        X.append(arr[i:i+L, 0])   # Sequence of length L
        y.append(arr[i+L, 0])     # Next value
    X = np.array(X, dtype=np.float32)  # [N, L]
    y = np.array(y, dtype=np.float32).reshape(-1, 1)  # [N, 1]
    return X, y

X_train, y_train = make_windows(close_train, look_back)
X_test,  y_test  = make_windows(close_test,  look_back)


In [ ]:
# -----------------------------
# 2) Torch Datasets / Loader
# -------------------------
class WindowDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

train_ds = WindowDataset(X_train, y_train)
test_ds  = WindowDataset(X_test, y_test)

batch_size = 64
train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=False)  # Time series: no shuffling
test_dl  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# -----------------------------
# 3) BayesianLinear Layer
#    (Blundell et al., 2015)
# -----------------------------
class BayesianLinear(nn.Module):
    def __init__(self, in_features, out_features, prior_sigma=1.0):
        super().__init__()
        # Variational posterior parameters (weights)
        self.w_mu   = nn.Parameter(torch.zeros(out_features, in_features))
        self.w_rho  = nn.Parameter(torch.full((out_features, in_features), -3.0))  # small std at initialization
        # Variational posterior parameters (bias)
        self.b_mu   = nn.Parameter(torch.zeros(out_features))
        self.b_rho  = nn.Parameter(torch.full((out_features,), -3.0))
        # Fixed prior
        self.prior_sigma = prior_sigma
        self.prior = torch.distributions.Normal(loc=0.0, scale=prior_sigma)
        # Standard normal distribution for sampling
        self.normal = torch.distributions.Normal(0, 1)

    def _softplus(self, x):
        return torch.log1p(torch.exp(x))  # numerically more stable than direct exp

    def sample_weights(self):
        w_sigma = self._softplus(self.w_rho)
        b_sigma = self._softplus(self.b_rho)
        eps_w = self.normal.sample(self.w_mu.shape).to(self.w_mu.device)
        eps_b = self.normal.sample(self.b_mu.shape).to(self.b_mu.device)
        w = self.w_mu + w_sigma * eps_w
        b = self.b_mu + b_sigma * eps_b
        return w, b, w_sigma, b_sigma

    def forward(self, x):
        # Weight sampling via reparameterization
        w, b, w_sigma, b_sigma = self.sample_weights()
        out = x @ w.t() + b  # [B, out_features]
        # KL divergence q||p for weights + bias (Gaussian-Gaussian, closed form)
        # KL(N(μ,σ)||N(0,σ0)) = log(σ0/σ) + (σ² + μ²)/(2σ0²) - 0.5
        prior_var = self.prior_sigma ** 2
        w_var = w_sigma**2
        b_var = b_sigma**2
        kl_w = torch.log(self.prior_sigma / w_sigma).sum() + 0.5 * ((w_var + self.w_mu**2) / prior_var - 1).sum()
        kl_b = torch.log(self.prior_sigma / b_sigma).sum() + 0.5 * ((b_var + self.b_mu**2) / prior_var - 1).sum()
        kl = kl_w + kl_b
        return out, kl


In [ ]:
# -----------------------------
# 4) BNN Model: MLP on windows
#    (simple, robust, independent of LSTM)
# -----------------------------
class BNN(nn.Module):
    def __init__(self, L, hidden=64, prior_sigma=1.0):
        super().__init__()
        self.fc1 = BayesianLinear(L, hidden, prior_sigma)  # First Bayesian linear layer
        self.act = nn.ReLU()                               # Activation function
        self.fc2 = BayesianLinear(hidden, 1, prior_sigma)  # Output Bayesian linear layer
        # Homoscedastic observation standard deviation (learnable, positive scalar)
        self.log_sigma_y = nn.Parameter(torch.tensor(-2.0))

    def forward(self, x):
        # x: [B, L]  (batch size, window length)
        h, kl1 = self.fc1(x)
        h = self.act(h)
        mu, kl2 = self.fc2(h)   # Predictive mean
        kl = kl1 + kl2           # Total KL divergence
        sigma_y = torch.nn.functional.softplus(self.log_sigma_y) + 1e-6
        return mu, sigma_y, kl


In [ ]:
# -----------------------------
# 5) Training (ELBO)
# -----------------------------

# Initialize BNN model and optimizer
model = BNN(L=look_back, hidden=64, prior_sigma=1.0).to(device)
optim = torch.optim.Adam(model.parameters(), lr=1e-3)

# --- Single ELBO step ---
def elbo_step(x, y, N):
    # Forward pass: stochastic due to weight sampling
    mu, sigma_y, kl = model(x)
    # Negative log-likelihood under Normal(mu, sigma_y)
    nll = 0.5*torch.log(2*math.pi*sigma_y**2) + 0.5*((y - mu)**2)/(sigma_y**2)
    nll = nll.mean()                  # Average over batch
    elbo = nll + kl / N               # ELBO: NLL + KL/N (Blundell et al., 2015)
    return elbo, nll.item(), kl.item(), sigma_y.item()

# Training loop
N = len(train_ds)
epochs = 80
for epoch in range(1, epochs+1):
    model.train()
    total_elbo, total_nll, total_kl = 0.0, 0.0, 0.0
    for xb, yb in train_dl:
        xb = xb.to(device)
        yb = yb.to(device)
        optim.zero_grad()
        elbo, nll_val, kl_val, _ = elbo_step(xb, yb, N)
        elbo.backward()
        optim.step()
        # Accumulate batch metrics
        total_elbo += elbo.item() * len(xb)
        total_nll  += nll_val     * len(xb)
        total_kl   += kl_val      * len(xb)
    # Print progress every 10 epochs or at first epoch
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | ELBO: {total_elbo/N:.4f} | NLL: {total_nll/N:.4f} | KL: {total_kl/N:.4f}")


Epoch   1 | ELBO: 0.8240 | NLL: 0.3514 | KL: 1932.9593
Epoch  10 | ELBO: -0.5722 | NLL: -1.0426 | KL: 1923.7362
Epoch  20 | ELBO: -0.6189 | NLL: -1.0873 | KL: 1915.5762
Epoch  30 | ELBO: -0.7923 | NLL: -1.2590 | KL: 1908.4499
Epoch  40 | ELBO: -0.8421 | NLL: -1.3073 | KL: 1902.3488
Epoch  50 | ELBO: -1.1261 | NLL: -1.5891 | KL: 1893.8408
Epoch  60 | ELBO: -1.0625 | NLL: -1.5228 | KL: 1882.5554
Epoch  70 | ELBO: -1.1869 | NLL: -1.6434 | KL: 1867.4292
Epoch  80 | ELBO: -1.3872 | NLL: -1.8400 | KL: 1851.8745


In [ ]:
# -----------------------------
# 6) MC Inference (T forward passes)
# -----------------------------
model.eval()
with torch.no_grad():
    Xte = torch.from_numpy(X_test).to(device)
    yte = torch.from_numpy(y_test).to(device)

    T = 100  # Number of Monte Carlo samples
    mu_samples = []
    sigma_obs = None

    # --- Perform T stochastic forward passes ---
    for _ in range(T):
        mu, sigma_y, _ = model(Xte)
        mu_samples.append(mu.cpu().numpy())
        sigma_obs = sigma_y.item()  # Homoscedastic: same for all passes

    # Convert to array: [T, N]
    mu_samples = np.stack(mu_samples, axis=0).squeeze(-1)

    # Compute predictive mean and epistemic variance
    mu_mean_scaled = mu_samples.mean(axis=0, keepdims=True).T    # [N,1]
    epistemic_var = mu_samples.var(axis=0, ddof=1, keepdims=True).T  # [N,1]

    # Aleatoric variance (homoscedastic)
    aleatoric_var = (sigma_obs**2) * np.ones_like(mu_mean_scaled)     # [N,1]

    # Total predictive standard deviation
    pred_std_scaled = np.sqrt(epistemic_var + aleatoric_var)

    # 95% confidence intervals (Gaussian approximation)
    q025 = mu_mean_scaled - 1.96 * pred_std_scaled
    q975 = mu_mean_scaled + 1.96 * pred_std_scaled


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# -----------------------------
# 7) Inverse scaling & metrics
# -----------------------------

# Inverse transform predictions and confidence intervals
y_test_inv = scaler.inverse_transform(y_test)
y_pred_inv = scaler.inverse_transform(mu_mean_scaled)
q025_inv   = scaler.inverse_transform(q025)
q975_inv   = scaler.inverse_transform(q975)

# Standard error metrics
rmse = math.sqrt(mean_squared_error(y_test_inv, y_pred_inv))
mae  = mean_absolute_error(y_test_inv, y_pred_inv)
r2   = r2_score(y_test_inv, y_pred_inv)

# Relative errors
mean_price = np.mean(y_test_inv)
rel_rmse = rmse / mean_price * 100
rel_mae = mae / mean_price * 100
mape = np.mean(np.abs((y_test_inv - y_pred_inv) / y_test_inv)) * 100

# 95% predictive interval coverage
coverage95 = np.mean((y_test_inv[:,0] >= q025_inv[:,0]) & (y_test_inv[:,0] <= q975_inv[:,0]))

# Print results
print("\n📊 BNN (Bayes-by-Backprop) Performance:")
print(f"RMSE: {rmse:.4f}   |  Rel. RMSE: {rel_rmse:.2f}%")
print(f"MAE:  {mae:.4f}   |  Rel. MAE:  {rel_mae:.2f}%")
print(f"MAPE: {mape:.2f}%")
print(f"R²:   {r2:.4f}")
print(f"95%-Coverage: {coverage95:.3f}")



📊 BNN (Bayes-by-Backprop) Performance:
RMSE: 5.0141   |  Rel. RMSE: 3.88%
MAE:  4.0707   |  Rel. MAE:  3.15%
MAPE: 3.20%
R²:   0.9622
95%-Coverage: 1.000


In [ ]:
"""
This code snippet was used to evaluate the Bayesian Neural Network model
on different historical time periods and save the performance metrics to a Word document.
"""

from docx import Document
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# -----------------------------
# 8) Save results to Word document (entire dataset)
# -----------------------------

# Inverse transform the entire dataset
close_inv = scaler.inverse_transform(close)

# Generate all windows + targets for the entire dataset
X_all, y_all = make_windows(scaler.transform(close), look_back)

# Convert to Torch format
X_all_torch = torch.from_numpy(X_all).to(device)

# MC inference on the entire dataset
model.eval()
with torch.no_grad():
    T = 100  # Number of MC samples
    mu_samples = []
    sigma_obs = None

    for _ in range(T):
        mu, sigma_y, _ = model(X_all_torch)
        mu_samples.append(mu.cpu().numpy())
        sigma_obs = sigma_y.item()

    mu_samples = np.stack(mu_samples, axis=0).squeeze(-1)  # [T, N]
    mu_mean_scaled = mu_samples.mean(axis=0, keepdims=True).T  # [N,1]
    epistemic_var = mu_samples.var(axis=0, ddof=1, keepdims=True).T
    aleatoric_var = (sigma_obs**2) * np.ones_like(mu_mean_scaled)
    pred_std_scaled = np.sqrt(epistemic_var + aleatoric_var)

# Inverse transform predictions to original scale
y_all_inv = scaler.inverse_transform(y_all)
mu_mean_inv = scaler.inverse_transform(mu_mean_scaled)

# Date column for the entire dataset
date_all = pd.to_datetime(df['Date'].values[look_back:])

# Helper function for metrics
def metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2 = r2_score(y_true, y_pred)
    rel_rmse = rmse / np.mean(y_true) * 100
    rel_mae  = mae / np.mean(y_true) * 100
    return mse, rmse, rel_rmse, mae, rel_mae, mape, r2

# === Define periods ===
periods = {
    "Entire period": ("2020-01-01", date_all.max()),
    "Financial Crisis 2008": ("2008-09-01", "2009-06-30"),
    "COVID-19 Crisis": ("2020-02-01", "2020-12-31"),
    "Russia-Ukraine War": ("2022-02-24", "2022-06-30"),
    "Stable Uptrend": ("2017-01-01", "2017-12-31")
}

# Create Word document
doc = Document()
doc.add_heading("Bayesian Neural Network Performance Results", level=1)

# === Calculate metrics & write to document ===
for name, (start, end) in periods.items():
    mask = (date_all >= pd.to_datetime(start)) & (date_all <= pd.to_datetime(end))
    if mask.sum() == 0:
        continue
    y_true = y_all_inv[mask]
    y_pred = mu_mean_inv[mask]
    mse, rmse, rel_rmse, mae, rel_mae, mape, r2 = metrics(y_true, y_pred)

    doc.add_heading(f"{name}", level=2)
    doc.add_paragraph(f"MSE: {mse:.4f}")
    doc.add_paragraph(f"RMSE: {rmse:.4f}  |  Rel. RMSE: {rel_rmse:.2f}%")
    doc.add_paragraph(f"MAE: {mae:.4f}   |  Rel. MAE: {rel_mae:.2f}%")
    doc.add_paragraph(f"MAPE: {mape:.2f}%")
    doc.add_paragraph(f"R²: {r2:.4f}")

# Save document
output_path = "BNN_Performance_AllPeriods2.docx"
doc.save(output_path)
print(f"\n✅ Results saved to: {output_path}")



✅ Ergebnisse wurden gespeichert in: BNN_Performance_AllPeriods2.docx
